In [90]:
import pandas as pd
import gc

In [91]:
fines = pd.read_csv("../ex04/fines.csv")
fines.head()

,CarNumber,Refund,Fines,Make,Model,Year
0,Y163O8161RUS,2,3200.0,Ford,Focus,1989
1,E432XX77RUS,1,6500.0,Toyota,Camry,1995
2,7184TT36RUS,1,2100.0,Ford,Focus,1984
3,X582HE161RUS,2,2000.0,Ford,Focus,2015
4,92918M178RUS,1,5700.0,Ford,Focus,2014


In [92]:
%%timeit
results = []
for i in range(len(fines)):
    val = fines.iloc[i]["Fines"] / fines.iloc[i]["Refund"] * fines.iloc[i]["Year"]
    results.append(val)

fines["Calc_for"] = results

73.5 ms ± 4.49 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [93]:
%%timeit
results = []
for _, row in fines.iterrows():
    results.append(row["Fines"] / row["Refund"] * row["Year"])

fines["Calc_iterrows"] = results

14.6 ms ± 269 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [94]:
%%timeit
fines["Calc_apply"] = fines.apply(lambda x: x["Fines"] / x["Refund"] * x["Year"], axis=1)

4.47 ms ± 175 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [95]:
%%timeit
fines["Calc_series"] = (fines["Fines"] / fines["Refund"]) * fines["Year"]

96.7 μs ± 3.38 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [96]:
%%timeit
fines["Calc_values"] = (fines["Fines"].values / fines["Refund"].values) * fines["Year"].values

41.8 μs ± 831 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [97]:
%%timeit
row = fines[fines["CarNumber"] == "O136HO197RUS"]

235 μs ± 14.8 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [98]:
fines_indexed = fines.set_index("CarNumber")

In [99]:
%%timeit
row = fines_indexed.loc["O136HO197RUS"]

37 μs ± 1.07 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [100]:
fines.info(memory_usage="deep")

<class 'pandas.DataFrame'>
RangeIndex: 930 entries, 0 to 929
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   CarNumber      930 non-null    str    
 1   Refund         930 non-null    int64  
 2   Fines          930 non-null    float64
 3   Make           930 non-null    str    
 4   Model          919 non-null    str    
 5   Year           930 non-null    int64  
 6   Calc_for       930 non-null    float64
 7   Calc_iterrows  930 non-null    float64
 8   Calc_apply     930 non-null    float64
 9   Calc_series    930 non-null    float64
 10  Calc_values    930 non-null    float64
dtypes: float64(6), int64(2), str(3)
memory usage: 232.9 KB


In [101]:
optimized_df = fines.copy()

In [102]:
# float64 → float32
for col in optimized_df.select_dtypes(include=["float64"]).columns:
    optimized_df[col] = pd.to_numeric(optimized_df[col], downcast="float")
    
# int64 → int32
for col in optimized_df.select_dtypes(include=["int64"]).columns:
    optimized_df[col] = pd.to_numeric(optimized_df[col], downcast="integer")

In [103]:
optimized_df.info(memory_usage="deep")

<class 'pandas.DataFrame'>
RangeIndex: 930 entries, 0 to 929
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   CarNumber      930 non-null    str    
 1   Refund         930 non-null    int8   
 2   Fines          930 non-null    float32
 3   Make           930 non-null    str    
 4   Model          919 non-null    str    
 5   Year           930 non-null    int16  
 6   Calc_for       930 non-null    float64
 7   Calc_iterrows  930 non-null    float64
 8   Calc_apply     930 non-null    float64
 9   Calc_series    930 non-null    float64
 10  Calc_values    930 non-null    float64
dtypes: float32(1), float64(5), int16(1), int8(1), str(3)
memory usage: 217.5 KB


In [104]:
# object_cols = optimized_df.select_dtypes(include=["object"]).columns
# object_cols
# Xullas pandas 4 versiyasida "object" turi o'rniga "str" turi qo'shilgan, shuning uchun str turini ham tekshirish kerak !!!

object_cols = optimized_df.select_dtypes(include=["str"]).columns
object_cols

Index(['CarNumber', 'Make', 'Model'], dtype='str')

In [105]:
optimized_df[object_cols] = optimized_df[object_cols].astype("category")

In [106]:
optimized_df.info(memory_usage="deep")

<class 'pandas.DataFrame'>
RangeIndex: 930 entries, 0 to 929
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype   
---  ------         --------------  -----   
 0   CarNumber      930 non-null    category
 1   Refund         930 non-null    int8    
 2   Fines          930 non-null    float32 
 3   Make           930 non-null    category
 4   Model          919 non-null    category
 5   Year           930 non-null    int16   
 6   Calc_for       930 non-null    float64 
 7   Calc_iterrows  930 non-null    float64 
 8   Calc_apply     930 non-null    float64 
 9   Calc_series    930 non-null    float64 
 10  Calc_values    930 non-null    float64 
dtypes: category(3), float32(1), float64(5), int16(1), int8(1)
memory usage: 83.6 KB


In [107]:
del fines
gc.collect()
%reset_selective -f fines